# exp007: A vs. B nas 15 categorias do MVTec AD (Kaggle)

Mesmo experimento do notebook do Colab, adaptado ao Kaggle. 740 imagens (50 por categoria, 40 no `toothbrush`), 3 perguntas, Configurações A e B.

Antes de rodar, no painel lateral do notebook:
- **Accelerator**: GPU T4 x2 (a P100 não tem bf16 e pode não ser suportada pelo torch atual).
- **Internet**: ligada (precisa para `git clone` e baixar o SmolVLM).
- **Input** (opcional): adicionar o dataset `ipythonx/mvtec-ad`. Sem ele, o `kagglehub` baixa.

Para rodar tudo sem manter a aba aberta, use **Save Version > Save & Run All (Commit)** (limite de 12 h). Os resultados ficam em `/kaggle/working` e aparecem na aba **Output** da versão, incluindo `exp007_results.zip`.

In [1]:
import os, subprocess, sys

STORAGE = '/kaggle/working'
os.makedirs(f'{STORAGE}/exp007_multicategoria', exist_ok=True)
os.makedirs(f'{STORAGE}/anomalib_runs', exist_ok=True)

def run(*args):
    # Com '!', um erro não para o notebook e as células seguintes rodam sobre arquivo faltando.
    subprocess.run([sys.executable, '-m', *args], check=True)

In [2]:
%cd /tmp
!test -d vision-explain || git clone https://github.com/Edryck/vision-explain.git
%cd /tmp/vision-explain
!git pull --ff-only
# torch e torchvision já vêm na imagem do Kaggle; reinstalar pode quebrar o CUDA.
!pip install -q "transformers>=4.45" accelerate anomalib scipy

/tmp
Cloning into 'vision-explain'...
remote: Enumerating objects: 258, done.
remote: Counting objects: 100% (258/258), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 258 (delta 107), reused 250 (delta 99), pack-reused 0 (from 0)
Receiving objects: 100% (258/258), 5.85 MiB | 31.69 MiB/s, done.
Resolving deltas: 100% (107/107), done.
/tmp/vision-explain
Already up to date.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Dados

Usa o dataset anexado em `/kaggle/input` se existir; senão baixa com `kagglehub` (cerca de 5 GB).

In [3]:
import kagglehub

if not os.path.isdir('data/mvtec_ad/mvtec-ad/bottle'):
    path = kagglehub.dataset_download('ipythonx/mvtec-ad')
    # O dataset pode vir com pasta extra; acha onde está o bottle.
    root = os.path.dirname(next(d for d, _, _ in os.walk(path) if d.endswith('/bottle')))
    !mkdir -p data/mvtec_ad
    !rm -rf data/mvtec_ad/mvtec-ad
    !ln -s {root} data/mvtec_ad/mvtec-ad
!ls data/mvtec_ad/mvtec-ad/

bottle	 carpet    leather	pill	    tile	wood
cable	 grid	   license.txt	readme.txt  toothbrush	zipper
capsule  hazelnut  metal_nut	screw	    transistor


Resultados e checkpoints do PatchCore ficam em `/kaggle/working` via link simbólico, para sair no Output da versão.

In [4]:
!test -f {STORAGE}/exp007_multicategoria/subset.json || cp experiments/exp007_multicategoria/subset.json {STORAGE}/exp007_multicategoria/
!rm -rf experiments/exp007_multicategoria data/anomalib_runs
!ln -s {STORAGE}/exp007_multicategoria experiments/exp007_multicategoria
!ln -s {STORAGE}/anomalib_runs data/anomalib_runs
!ls experiments/exp007_multicategoria

subset.json


In [5]:
import torch
print(torch.cuda.device_count(), 'GPU(s):', torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory / 1024**3, 'GB')
print('bf16:', torch.cuda.is_bf16_supported())

2 GPU(s): Tesla T4 14.56219482421875 GB
bf16: True


## 1. Evidências do PatchCore

Treina um PatchCore por categoria (coreset 0,01), roda o teste completo e extrai a evidência das imagens do subconjunto.

In [6]:
run('src.configB.exp007_multicategoria.evidences')

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/anomalib/models/image/dinomaly/components/layers.py:22: FutureWarning: The anomalib.models.components.dinov2 package is deprecated and will be removed in a future release. Please use the timm-based feature extractor instead: anomalib.models.components.feature_extractors.TimmFeatureExtractor
  from anomalib.models.components.dinov2.layers import Attention, DropPath, LayerScale, MemEffAttention
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning E

┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor   │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor  │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator      │      0 │ train │     0 │
│ 3 │ model          │ PatchcoreModel │ 24.9 M │ train │     0 │
└───┴────────────────┴────────────────┴────────┴───────┴───────┘
Trainable params: 24.9 M                                                        
Non-trainable params: 0                                                         
Total params: 24.9 M                                                            
Total estimated model params size (MB): 99.450                                  
Modules in train mode: 19                                                       
Modules in eval mode: 174                                                  

Selecting Coreset Indices.: 100%|██████████| 2139/2139 [00:07<00:00, 296.52it/s]
`Trainer.fit` stopped: `max_epochs=1` reached.
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
Restoring states from the checkpoint path at /kaggle/working/anomalib_runs/Patchcore/MVTecAD/bottle/v0/weights/lightning/model.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /kaggle/working/anomalib_runs/Patchcore/MVTecAD/bottle/v0/weights/lightning/model.ckpt


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │            1.0            │
│       image_F1Score       │    0.9919999837875366     │
│        pixel_AUROC        │    0.9852288365364075     │
│       pixel_F1Score       │    0.7243679761886597     │
└───────────────────────────┴───────────────────────────┘


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
Restoring states from the checkpoint path at /kaggle/working/anomalib_runs/Patchcore/MVTecAD/bottle/v0/weights/lightning/model.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /kaggle/working/anomalib_runs/Patchcore/MVTecAD/bottle/v0/weights/lightning/model.ckpt
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
Restoring states from the 

bottle {'image_AUROC': 1.0, 'image_F1Score': 0.992, 'pixel_AUROC': 0.9852, 'pixel_F1Score': 0.7244}


... (80 outputs left out)

Selecting Coreset Indices.: 100%|██████████| 2456/2456 [00:10<00:00, 244.43it/s]
`Trainer.fit` stopped: `max_epochs=1` reached.
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
Restoring states from the checkpoint path at /kaggle/working/anomalib_runs/Patchcore/MVTecAD/zipper/v0/weights/lightning/model.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /kaggle/working/anomalib_runs/Patchcore/MVTecAD/zipper/v0/weights/lightning/model.ckpt


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │    0.9824054837226868     │
│       image_F1Score       │    0.9831932783126831     │
│        pixel_AUROC        │    0.9813777208328247     │
│       pixel_F1Score       │    0.5421947240829468     │
└───────────────────────────┴───────────────────────────┘


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
Restoring states from the checkpoint path at /kaggle/working/anomalib_runs/Patchcore/MVTecAD/zipper/v0/weights/lightning/model.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /kaggle/working/anomalib_runs/Patchcore/MVTecAD/zipper/v0/weights/lightning/model.ckpt
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
Restoring states from the 

zipper {'image_AUROC': 0.9824, 'image_F1Score': 0.9832, 'pixel_AUROC': 0.9814, 'pixel_F1Score': 0.5422}


## 2. Configuração A (imagem + pergunta)

In [7]:
run('src.configA.exp007_multicategoria.run')

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/anomalib/models/image/dinomaly/components/layers.py:22: FutureWarning: The anomalib.models.components.dinov2 package is deprecated and will be removed in a future release. Please use the timm-based feature extractor instead: anomalib.models.components.feature_extractors.TimmFeatureExtractor
  from anomalib.models.components.dinov2.layers import Attention, DropPath, LayerScale, MemEffAttention
Loading weights: 100%|██████████| 657/657 [00:00<00:00, 2636.80it/s]


3 2220 bottle_good_000 6.5
6 2220 bottle_good_001 6.3
9 2220 bottle_good_002 6.2
12 2220 bottle_good_003 6.2
15 2220 bottle_good_007 6.3
18 2220 bottle_good_008 6.3
21 2220 bottle_good_010 6.9
24 2220 bottle_good_011 6.1
27 2220 bottle_good_016 6.0
30 2220 bottle_good_017 5.9
33 2220 bottle_broken_large_000 6.2
36 2220 bottle_broken_large_001 5.8
39 2220 bottle_broken_large_002 5.9
42 2220 bottle_broken_large_003 6.3
45 2220 bottle_broken_large_005 6.2
48 2220 bottle_broken_large_008 6.5
51 2220 bottle_broken_large_009 6.3
54 2220 bottle_broken_large_011 6.0
57 2220 bottle_broken_large_012 6.1
60 2220 bottle_broken_large_013 6.4
63 2220 bottle_broken_large_015 6.2
66 2220 bottle_broken_large_016 6.1
69 2220 bottle_broken_large_017 6.3
72 2220 bottle_broken_large_018 6.4
75 2220 bottle_broken_small_000 6.3
78 2220 bottle_broken_small_001 6.5
81 2220 bottle_broken_small_002 6.2
84 2220 bottle_broken_small_003 6.1
87 2220 bottle_broken_small_005 6.3
90 2220 bottle_broken_small_008 6.4
93 

## 3. Configuração B (imagem + evidência + pergunta)

In [8]:
run('src.configB.exp007_multicategoria.run')

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/anomalib/models/image/dinomaly/components/layers.py:22: FutureWarning: The anomalib.models.components.dinov2 package is deprecated and will be removed in a future release. Please use the timm-based feature extractor instead: anomalib.models.components.feature_extractors.TimmFeatureExtractor
  from anomalib.models.components.dinov2.layers import Attention, DropPath, LayerScale, MemEffAttention
Loading weights: 100%|██████████| 657/657 [00:00<00:00, 2575.15it/s]


3 2220 bottle_good_000 6.3
6 2220 bottle_good_001 5.9
9 2220 bottle_good_002 5.9
12 2220 bottle_good_003 5.9
15 2220 bottle_good_007 5.9
18 2220 bottle_good_008 5.9
21 2220 bottle_good_010 6.0
24 2220 bottle_good_011 5.9
27 2220 bottle_good_016 5.9
30 2220 bottle_good_017 6.0
33 2220 bottle_broken_large_000 6.8
36 2220 bottle_broken_large_001 5.9
39 2220 bottle_broken_large_002 5.8
42 2220 bottle_broken_large_003 5.9
45 2220 bottle_broken_large_005 6.9
48 2220 bottle_broken_large_008 5.9
51 2220 bottle_broken_large_009 5.9
54 2220 bottle_broken_large_011 6.1
57 2220 bottle_broken_large_012 6.1
60 2220 bottle_broken_large_013 5.9
63 2220 bottle_broken_large_015 6.3
66 2220 bottle_broken_large_016 5.9
69 2220 bottle_broken_large_017 5.9
72 2220 bottle_broken_large_018 6.0
75 2220 bottle_broken_small_000 5.9
78 2220 bottle_broken_small_001 6.0
81 2220 bottle_broken_small_002 5.9
84 2220 bottle_broken_small_003 6.0
87 2220 bottle_broken_small_005 6.1
90 2220 bottle_broken_small_008 5.9
93 

## 4. Métricas

Uma célula por passo, para saber qual falhou. O FaithScore carrega o SmolVLM de novo; se faltar memória, reinicie o kernel e rode só as células de setup e de métricas.

In [9]:
run('src.multicategory_metrics', 'score', '--config', 'A', '--faithscore')

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/anomalib/models/image/dinomaly/components/layers.py:22: FutureWarning: The anomalib.models.components.dinov2 package is deprecated and will be removed in a future release. Please use the timm-based feature extractor instead: anomalib.models.components.feature_extractors.TimmFeatureExtractor
  from anomalib.models.components.dinov2.layers import Attention, DropPath, LayerScale, MemEffAttention
Loading weights: 100%|██████████| 657/657 [00:00<00:00, 2725.68it/s]


50 2220
100 2220
150 2220
200 2220
250 2220
300 2220
350 2220
400 2220
450 2220
500 2220
550 2220
600 2220
650 2220
700 2220
750 2220
800 2220
850 2220
900 2220
950 2220
1000 2220
1050 2220
1100 2220
1150 2220
1200 2220
1250 2220
1300 2220
1350 2220
1400 2220
1450 2220
1500 2220
1550 2220
1600 2220
1650 2220
1700 2220
1750 2220
1800 2220
1850 2220
1900 2220
1950 2220
2000 2220
2050 2220
2100 2220
2150 2220
2200 2220
experiments/exp007_multicategoria/results/configA_pontuado.json 2220


In [10]:
run('src.multicategory_metrics', 'score', '--config', 'B', '--faithscore')

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/anomalib/models/image/dinomaly/components/layers.py:22: FutureWarning: The anomalib.models.components.dinov2 package is deprecated and will be removed in a future release. Please use the timm-based feature extractor instead: anomalib.models.components.feature_extractors.TimmFeatureExtractor
  from anomalib.models.components.dinov2.layers import Attention, DropPath, LayerScale, MemEffAttention
Loading weights: 100%|██████████| 657/657 [00:00<00:00, 2599.61it/s]


50 2220
100 2220
150 2220
200 2220
250 2220
300 2220
350 2220
400 2220
450 2220
500 2220
550 2220
600 2220
650 2220
700 2220
750 2220
800 2220
850 2220
900 2220
950 2220
1000 2220
1050 2220
1100 2220
1150 2220
1200 2220
1250 2220
1300 2220
1350 2220
1400 2220
1450 2220
1500 2220
1550 2220
1600 2220
1650 2220
1700 2220
1750 2220
1800 2220
1850 2220
1900 2220
1950 2220
2000 2220
2050 2220
2100 2220
2150 2220
2200 2220
experiments/exp007_multicategoria/results/configB_pontuado.json 2220


In [11]:
run('src.multicategory_metrics', 'report')

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/anomalib/models/image/dinomaly/components/layers.py:22: FutureWarning: The anomalib.models.components.dinov2 package is deprecated and will be removed in a future release. Please use the timm-based feature extractor instead: anomalib.models.components.feature_extractors.TimmFeatureExtractor
  from anomalib.models.components.dinov2.layers import Attention, DropPath, LayerScale, MemEffAttention


experiments/exp007_multicategoria/results/resumo_exp007.json


In [12]:
import pandas as pd
pd.read_csv('experiments/exp007_multicategoria/results/tabela_por_categoria.csv')

,categoria,image_auroc,pixel_auroc,evidencia_correta,acerto_A,acerto_B,q1_A,q1_B,q2_A,q2_B,q3_A,q3_B,inconsistencia_A,inconsistencia_B,tokens_A,tokens_B,mcnemar_p
0,bottle,1.0000,0.9852,1.000,0.5703,0.7383,0.9091,0.9600,0.6444,1.0000,0.28,0.260,0.2121,0.0400,7.5667,4.8067,0.009475
1,cable,0.9916,0.9830,0.960,0.2500,0.6434,0.8571,0.9388,0.4857,0.8182,0.00,0.200,0.2857,0.1633,5.2933,6.8067,0.000040
2,capsule,0.9908,0.9904,0.620,0.3619,0.3819,0.7778,0.3200,0.3478,0.6364,0.30,0.220,0.0000,0.4800,8.3733,4.4467,0.326940
3,carpet,0.9848,0.9902,0.880,0.5959,0.7267,0.8696,1.0000,0.6600,0.9200,0.28,0.260,0.1739,0.1400,7.2333,4.1400,0.000277
4,grid,0.9774,0.9772,0.820,0.3759,0.6892,0.5610,0.9600,0.4200,0.7708,0.18,0.340,0.6341,0.1400,5.5200,4.5667,0.000000
5,hazelnut,1.0000,0.9876,0.980,0.5680,0.7222,0.8205,0.8600,0.5556,0.9091,0.38,0.420,0.2051,0.3400,8.5867,3.9600,0.000535
6,leather,1.0000,0.9922,0.840,0.4755,0.7114,0.9535,1.0000,0.5000,0.9184,0.04,0.220,0.2791,0.0000,7.9067,5.4533,0.000000
7,metal_nut,1.0000,0.9862,0.960,0.3525,0.7114,0.6000,0.9800,0.4898,0.9592,0.02,0.200,0.4750,0.0600,6.6533,5.0933,0.000000
8,pill,0.9591,0.9817,0.940,0.3379,0.5733,0.4792,0.5800,0.2979,0.9200,0.24,0.220,0.5625,0.4200,6.7467,5.4667,0.000019
9,screw,0.9670,0.9804,0.560,0.3546,0.5168,0.6383,0.7600,0.2500,0.5918,0.18,0.200,0.4468,0.2600,8.8800,5.0800,0.000066


In [13]:
import json
summary = json.load(open('experiments/exp007_multicategoria/results/resumo_exp007.json'))
print(json.dumps({'overall': summary['overall'], 'paired': summary['paired']}, indent=1)[:6000])

{
 "overall": {
  "A": {
   "ground_truth_correct": {
    "n": 1966,
    "n_undefined": 254,
    "mean": 0.439,
    "std": 0.4964,
    "median": 0.0,
    "min": 0.0,
    "max": 1.0
   },
   "by_question": {
    "q1_presence": {
     "n": 546,
     "n_undefined": 194,
     "mean": 0.6941,
     "std": 0.4612,
     "median": 1.0,
     "min": 0.0,
     "max": 1.0
    },
    "q2_location": {
     "n": 680,
     "n_undefined": 60,
     "mean": 0.4882,
     "std": 0.5002,
     "median": 0.0,
     "min": 0.0,
     "max": 1.0
    },
    "q3_type": {
     "n": 740,
     "n_undefined": 0,
     "mean": 0.2054,
     "std": 0.4043,
     "median": 0.0,
     "min": 0.0,
     "max": 1.0
    }
   },
   "inconsistent_images": {
    "n": 546,
    "n_undefined": 194,
    "mean": 0.3755,
    "std": 0.4847,
    "median": 0.0,
    "min": 0.0,
    "max": 1.0
   },
   "richness": {
    "new_tokens": {
     "n": 2220,
     "n_undefined": 0,
     "mean": 7.1,
     "std": 4.935,
     "median": 5.0,
     "min": 2.0

## 5. Exportar resultados

Gera `/kaggle/working/exp007_results.zip` sem os checkpoints do PatchCore (pesados). Baixe pela aba Output e extraia em `experiments/exp007_multicategoria/` no repo local.

In [14]:
import shutil
shutil.make_archive(f'{STORAGE}/exp007_results', 'zip', f'{STORAGE}/exp007_multicategoria')
print(os.path.getsize(f'{STORAGE}/exp007_results.zip') / 1e6, 'MB')

0.191933 MB
